### Preliminary cell to start the notebook

In [ ]:
# libraries
import os
import sys
import platform

print(sys.version)

strong_pc = platform.system() == "Linux"
in_colab = "google.colab" in sys.modules
if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    #!pip install tensorflow==2.11.0
    #!pip install tensorflow_text==2.11.0
    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules and False:
        print("Installing keras")
        !pip install keras==2.11.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0


if "DEEPNOTE_ENV" in os.environ:
    os.chdir("/..")
    os.chdir("datasets")
    os.chdir("googledrivedeepnoteintegration")
    os.chdir("Human_Data_Analytics_Project_2023")
    if not "librosa" in sys.modules:
        print("Installing Librosa")
        !pip install librosa
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)

# BASE LIBRARIES
import numpy as np
import pandas as pd
import h5py
import shutil
import time
import random
import subprocess
import itertools
import warnings
import pickle
import json

# PLOT LIBRARIES
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
import IPython.display as ipd

# import plotly.express as px

# AUDIO LIBRARIES
import librosa
from scipy.io import wavfile
from scipy import signal
from scipy.fft import fft, ifft, fftfreq, fftshift
from scipy.signal import stft, spectrogram, periodogram

# from pydub import AudioSegment

# MACHINE LEARNING LIBRARIES
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.utils import check_random_state
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras.models import load_model

# GPU SETTINGS FOR LINUX and repressing warnings for windows. References for gpu: https://www.tensorflow.org/guide/gpu
show_gpu_activity = False
if sys.platform == "linux" and not in_colab:
    if show_gpu_activity:
        tf.debugging.set_log_device_placement(True)

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        # Restrict TensorFlow to only allocate a part of memory on the first GPU
        try:
            tf.config.set_logical_device_configuration(
                gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6800)]
            )
            logical_gpus = tf.config.list_logical_devices("GPU")
            print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
        except RuntimeError as e:
            # Virtual devices must be set before GPUs have been initialized
            print(e)
else:
    warnings.filterwarnings("ignore", category=UserWarning)

from keras import layers
from keras import models
from keras.utils import plot_model as tf_plot

if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
# show keras version
import keras

print(f"keras version = {keras.__version__}")
# import keras_tune as kt
from keras import layers
import keras_tuner as kt
from tensorflow import keras
from keras.regularizers import L1L2

# kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4) # we may use this in some layers...

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# EVALUATION LIBRAIRES
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
)
from sklearn.metrics import precision_recall_fscore_support, auc

# OUR PERSONAL FUNCTIONS
import importlib
from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    one_random_audio,
    plot_clip_overview,
    Spectral_Analysis,
)
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Preprocessing.data_loader import load_metadata

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)

from Preprocessing.data_loader import load_metadata
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer
from Visualization.model_plot import (
    plot_history,
    confusion_matrix,
    listen_to_wrong_audio,
    visualize_the_weights,
)

ESC10_path = os.path.join(main_dir, "Data", "ESC-10-depth")
samplerate = 44100

# **Environmetal sound classification**


<a href="https://github.com/GianmarcoLattaruolo/Human_Data_Analytics_Project_2023/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here we import the drive (in the case in which we are running the notebook on Colab) and prepare the environment.

In [ ]:
import os
import sys
import platform

print(sys.version)

strong_pc = platform.system() == "Linux"
in_colab = "google.colab" in sys.modules
if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    #!pip install tensorflow==2.11.0
    #!pip install tensorflow_text==2.11.0
    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules and False:
        print("Installing keras")
        !pip install keras==2.11.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0


if "DEEPNOTE_ENV" in os.environ:
    os.chdir("/..")
    os.chdir("datasets")
    os.chdir("googledrivedeepnoteintegration")
    os.chdir("Human_Data_Analytics_Project_2023")
    if not "librosa" in sys.modules:
        print("Installing Librosa")
        !pip install librosa
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)


# GPU SETTINGS FOR LINUX and repressing warnings for windows. References for gpu: https://www.tensorflow.org/guide/gpu
show_gpu_activity = False
if sys.platform == "linux" and not in_colab:
    if show_gpu_activity:
        tf.debugging.set_log_device_placement(True)

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        # Restrict TensorFlow to only allocate a part of memory on the first GPU
        try:
            tf.config.set_logical_device_configuration(
                gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6800)]
            )
            logical_gpus = tf.config.list_logical_devices("GPU")
            print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
        except RuntimeError as e:
            # Virtual devices must be set before GPUs have been initialized
            print(e)
else:
    warnings.filterwarnings("ignore", category=UserWarning)

In this cell, we import the libraries employed in our study, along with various custom scripts developed to enhance code readability. These scripts are accessible within our GitHub repository, organized into three principal directories:

-   **Preprocessing**:
    Within the "Preprocessing" directory, we provide the following scripts:
    -   `data_loader.py``: This script facilitates dataset acquisition, file integrity verification, and automatic distribution of files into appropriate subfolders. Note that we will not use commands from this script in the notebook as they serve only once, during the initial setup.
    -   `exploration_plots.py`: Designed for dataset visualization and audio file spectrum analysis, this script capitalizes on the capabilities of the `librosa` library. Multiple spectrograms are generated and explored within this context.
-   **Visualization**:
    The "Visualization" directory contains the `model_plot.py` script, which incorporates various utility functions we devised to visualize model outcomes. These functions encompass the presentation of the confusion matrix, history plots, weight and latent space visualizations, comparison to show the reconstruction capabilities of the autoencoders , as well as auditory playback functions.
-   **Models**:
    Our "Models" directory encompasses the following scripts:
    -   `basic_ml.py`: This script implements fundamental machine learning models, which are logistic regression, random forest, decision trees, k-nearest neighbors, and support vector machines. Both grid search procedures and training routines are integrated within this script to avoid a focused treatment of this non-neural model category. Indeed it's important to clarify that our primary emphasis does not lie in these basic models; rather, they are implemented for comparative reference.
    -   `ann_utils.py`: At the heart of our implementations lies the 'ann_utils' script. This script defines crucial functions for a wide array of tasks, such as constructing TensorFlow datasets with assorted preprocessing methodologies, creating, training, and evaluating models, as well as visualizing results. Of particular note is our exploration of grid search techniques, involving study and experimentation with two libraries: `keras-tuner` and `scikeras`.

By structuring our work in this manner, we hope to provide a clear framework for the execution, visualization, and evaluation of our methodologies. 

In [ ]:
# BASE LIBRARIES
import numpy as np
import pandas as pd
import shutil
import time
import random
import warnings
import pickle
import json

# PLOT LIBRARIES
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
import IPython.display as ipd  # for the listen to audio files

# AUDIO LIBRARIES
import librosa

# BASIC MACHINE LEARNING LIBRARIES
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.utils import check_random_state

# DEEP LEARNING LIBRARIES
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras.models import load_model
from keras import layers
from keras import models
from keras.utils import plot_model as tf_plot
import keras
from keras import layers
import keras_tuner as kt
from tensorflow import keras
from keras.regularizers import L1L2

if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
print(f"keras version = {keras.__version__}")

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# EVALUATION LIBRAIRES
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
)
from sklearn.metrics import precision_recall_fscore_support, auc

# OUR DEPENDENCIES (with reload to update the changes)
import importlib

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Preprocessing.exploration_plots"))
importlib.reload(importlib.import_module("Visualization.model_plot"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Models.ann_utils"))

from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    one_random_audio,
    plot_clip_overview,
    Spectral_Analysis,
)

from Visualization.model_plot import (
    confusion_matrix,
    plot_history,
    confusion_matrix,
    listen_to_wrong_audio,
    visualize_the_weights,
)

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)
main_dir = os.getcwd()
ESC10_path = os.path.join(main_dir, "Data", "ESC-10-depth")
samplerate = 44100

# 1 EXPLORATORY ANALYSIS

In this chapter we are going to explore the dataset and the audio files. We will also perform some basic preprocessing operations related to the Spectrum Analysis of the audio files.

## 1.1 Preliminary Exploration of the dataset

Here we can listen to one random audio in our dataset (different every time we run the cell).
Moreover we visualize the audio as one-dimesional plot of a function of time.


In [ ]:
import importlib

importlib.reload(importlib.import_module("Preprocessing.exploration_plots"))
from Preprocessing.exploration_plots import one_random_audio

audio, samplerate = one_random_audio(main_dir, end=220500)

The information about the dataset, including the file name, the file path, the class and the belonging to the ESC-10 or ESC-50 dataset are loaded in the following cell.

In [ ]:
importlib.reload(importlib.import_module("Preprocessing.data_loader"))
from Preprocessing.data_loader import load_metadata

df_ESC10, df_ESC50 = load_metadata(main_dir, ESC_US=False, statistics=False)

## 1.2 Spectrogram Analysis

Now we dig in the analysis, exploiting the natural structure of sound.
In particular we'll need the [Short-Time Fourier Transform](https://en.wikipedia.org/wiki/Short-time_Fourier_transform) and several other spectral transformations.

In [ ]:
importlib.reload(importlib.import_module("Preprocessing.exploration_plots"))
from Preprocessing.exploration_plots import Spectral_Analysis

mfcc_audio = Spectral_Analysis(
    audio,
    sample_rate=44100,
    segment=40,  # length of the segment in ms
    n_fft=None,  # padd the frames with zeros before DFT
    overlapping=5,
    cepstral_num=40,  # number of mel frequencies cepstral coefficients
    N_filters=50,  # number of mel filters in frequency domain
    plot=True,  # default is false
    verbose=False,  # default is false
    STFT_decibel=False,
    Mel_spectrogram_decibel=False,
    MFCC=True,
)

From the prevoius analysis we can see that the most informative plots, from a human point of view, are the one produced by using STFT, MEL or MFCC transformation of the audio. Here there are 5 plots of preprocessed audio (STFTs of audio converted into decibel domain and in logarithm scale) for each class of our dataset. Changing the preprocessing parameters we can obtain different plots.

In [ ]:
importlib.reload(importlib.import_module("Preprocessing.exploration_plots"))
from Preprocessing.exploration_plots import plot_clip_overview

plot_clip_overview(
    df_ESC10, preprocessing="STFT"
)  # clearly you can pass only df_ESC10 or df_ESC50

We can appreciate a ramarkable amount of intra-class similarity but also a non trascurable amount of inter-class similarity.

# 2 SUPERVISED LEARNING

In this chapter we are going to perform some supervised learning on the dataset. We will use the basic machine learning models and the neural networks. Of particular interest and not without difficulty was the creation of a routine for the dataset suitable for our purposes. In particular:
-   In the basic ML section it was sufficient for us to convert the audios into numpy arrays, which in the case of the 50-class dataset means an array of size 2000 x 220500. 
This is already beyond the limits of our ram possibilities, but can easily be solved by changing the format for saving values from float64 to float16.
-   In the section on neural networks, with a view to reusing the same function to create much larger datasets for autoencoders (up to 250000 audio as opposed to 2000 for the dataset with labels) we followed the typical processing pipeline of tensorflow, which was also shown in the course labs.

## 2.1 Basic Machine Learning

As said  before, since our interest here is not in exhaustive research ofthe best non-neuroal machine learning models, but just in having a base line for our research, we are going to build Numpy arrays with our raw audio or with the matrix of MFCC flattend (avoiding the use of other preprocessing types).

We will use this strategy to implement the following basic machine-learning models:
- Logistic Regression
- SVM
- Decision Tree and Random Forest
- KNN

We will experiment  setting a 1/4 of the dataset as test set. To understand if the 10-class dataset is actually made of classes "well separeted" we also built another 10-classes dataset (with 10 random classes among the 50 available) to have another comparison of the performance of the models.

In [ ]:
importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))
from Preprocessing.data_loader import load_metadata
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

Here we create the datasets as numpy arrays.

In [ ]:
X_ESC10, y_ESC10, labels10 = build_dataset(df_ESC10)
X_ESC50, y_ESC50, labels50 = build_dataset(df_ESC50)
df_subset, X_subset, y_subset, labels_subset = build_dataset(
    df_ESC50, subset=True, num_classes=10
)  # build a dataset of num_classes classes choosen among the 50 available

This function runs several grid searches to find the best hyperparameters for the models.
The results are written in the file "grid_search_results.txt" in the "Models" folder.

In [ ]:
basic_ML_experiments_gridsearch(
    X_ESC10,
    y_ESC10,
    test_size=0.25,
    file=r"Models\grid_search_results.txt",  # where to save the results
    cv=3,  # number of fold of the cross validation
    verbose=False,
    logistic_regression=False,  # which models to run
    SVM=False,
    decision_tree=False,
    random_forest=False,
    KNN=False,
)

Now, for the different datasets created before, we trained the models with the best hyperparameters found in the grid search. The cell output concisely shows the results.

In [ ]:
results_ESC10 = basic_ML_experiments(
    X_ESC10,
    y_ESC10,
    test_size=0.25,
    MFCC=True,
    raw_audio=True,
    logistic_regression=True,
    SVM=True,
    decision_tree=True,
    random_forest=True,
    KNN=True,
)

In [ ]:
# test on 10 random classes and not with the 10 well separated of ESC10
results_subset = basic_ML_experiments(X_subset, y_subset)

In [ ]:
results_ESC50 = basic_ML_experiments(X_ESC50, y_ESC50)

Here the results are summarized in a table. We can see that the best model is the SVM, which in the next cell we are going to build and train again to show a better visualization of the results.

In [ ]:
results_ESC50

In [ ]:
# show only the best model
mfcc = 50  # 40, 30
N_filters = 160  # 80,40
C = 100  # 10, 1
kernel = "rbf"
# kernel = 'poly'
# gamma = 1
# degree = 2
test_size = 0.25
name_df = "ESC50"

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X_ESC50, y_ESC50, test_size=test_size, random_state=123
)

# mfcc extraction
start_time = time.time()
X_train_mfcc = np.apply_along_axis(
    lambda x: extract_flatten_MFCC(audio=x, cepstral_num=mfcc, N_filters=N_filters),
    1,
    X_train,
)
X_test_mfcc = np.apply_along_axis(
    lambda x: extract_flatten_MFCC(audio=x, cepstral_num=mfcc, N_filters=N_filters),
    1,
    X_test,
)
print(f"Extract the MFCC requires {round(time.time()-start_time,2)} seconds.")

# fit the model
start_time = time.time()
pipe = make_pipeline(StandardScaler(), SVC(C=C, kernel=kernel, random_state=123))
pipe.fit(X_train_mfcc, y_train)
print(
    f"Fit the SVM on the {name_df} dataset with {mfcc}-MFCC and {N_filters} filters requires {round(time.time()-start_time,2)} seconds"
)

# accuracy on train and test
y_predict_train = pipe.predict(X_train_mfcc)
y_predict_test = pipe.predict(X_test_mfcc)
print(
    f"(SVM) Accuracy on {name_df} training set with {mfcc}-MFCC and {N_filters} filters. \t: {accuracy_score(y_train, y_predict_train)}"
)
print(
    f"(SVM) Accuracy on {name_df} test set with {mfcc}-MFCC and {N_filters} filters.\t: {accuracy_score(y_test, y_predict_test)}"
)

# confusion matrix
confusion_mtx = confusion_matrix(y_test, y_predict_test, labels50)

# listen to wrong audio
listen_to_wrong_audio(
    df_ESC50, y_test, y_predict_test, confusion_mtx=confusion_mtx, labels=labels50
)  # , n_audio = 2)

## 2.2 Fully Connected Neural Networks

In this section, we will proceed with the construction and training of multiple fully connected neural networks. It is important to note that our motivation for doing so is primarily comparative. This decision arises from the understanding that employing a Feedforward Neural Network (FFNN) with 220,500 variables while having only 40 samples per class is not prudent due to the limitations posed by such a scenario.

Furthermore, in the context of the dataset consisting of 50 classes, the feasibility of conducting a grid search is severely restricted. To address this, we have opted to utilize the optimal parameter configuration derived from the grid search conducted on the 10-class dataset. We intend to employ a similar approach for the majority of the subsequent models. From our perspective, this strategy is well-founded. It is based on the premise that the most suitable model configuration should remain consistent across both datasets, as it is contingent upon the inherent characteristics of the audio content rather than the quantity of classes present.

### 2.2.1 Fully connected - Raw Audio - 10 classes

In [ ]:
importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    create_dataset,
    compile_and_fit,
    example_batch,
    compile_fit_evaluate,
    K_fold_training,
)

The subsequent procedure will be replicated several times throughout our examination of each specific class of models and will not be commented any more:

1.  Creation of the dataset, encompassing all requisite preprocessing steps (for this instance, we utilize the raw audio data).
2.   Design and instantiation of the model, typically facilitated through a dedicated function often denoted as build_model.
3.   Execution of a Grid Search aimed at identifying the optimal hyperparameters (in this chapter we'll employ the scikeras library).
4.   Construction of the model with the most favorable hyperparameter configuration as determined by the Hyperparameter Optimization (HPO) process and subsequent training and evaluation. It is worth noting that this step will generally involve a more data than the preceding stages, especiallt in for the Autoencoder models for which we had almost 250000 audio files.

This entire procedure is executed using the ESC-10 dataset. Subsequently, for the ESC-50 dataset, steps 2 and 4 are omitted, given that the optimal hyperparameters are already established (as said before).

#### Create the dataset

In [ ]:
batch_size = 30

dataset, label = create_dataset_lite(
    df_ESC10, batch_size=batch_size, preprocessing=None, ndim=1
)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns))

#### Build the model

In [ ]:
def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=[64],
    activation="relu",
    dropout=0.5,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=(
        tf.keras.optimizers.legacy.Adam(learning_rate=1e-3)
        if (sys.platform == "darwin" or in_colab)
        else tf.keras.optimizers.Adam(learning_rate=1e-3)
    ),
    metrics=["accuracy"],
):

    model = tf.keras.models.Sequential(
        [tf.keras.layers.Flatten(input_shape=INPUT_DIM)], name="Flatten_ANN"
    )
    for unit in n_units:
        model.add(tf.keras.layers.Dense(unit, activation=activation))
        model.add(tf.keras.layers.Dropout(dropout))
    model.add(tf.keras.layers.Dense(n_labels, activation="softmax"))

    model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

    return model

#### Run a grid search to find the best params

In [ ]:
epochs = 100
patience = 20
params = {
    "n_units": [[64], [128], [64, 128], [8, 8, 16]],
    "activation": ["relu", "tanh", "elu"],
    "dropout": [0.5, 0.3, 0.1],
}
K_fold = 5

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=1,
    K=K_fold,
)

In [ ]:
# best_params = {'activation': 'relu', 'dropout': 0.3, 'n_units': [64, 128]}
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle

with open(
    os.path.join(main_dir, "Models", "best_params_RAW_FC_10c_221.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the model with best HP

In [ ]:
# best_params = {'activation': 'relu', 'n_units': 64}	#best parameters from the grid search
ESC10_path = os.path.join(main_dir, "data", "ESC-10-depth")
batch_size = 30
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC10_path,
    batch_size=batch_size,
    shuffle=True,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    show_example_batch=True,
    verbose=0,
)

In [ ]:
model = build_model(**best_params)

epochs = 100
patience = 20
# steps_per_epoch = np.ceil(300/batch_size) #needed only if we used the repeat method
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC10,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    verbose=0,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

### 2.2.2 Fully Connected - Raw Audio - 50 classes

In [ ]:
importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    create_dataset,
    compile_and_fit,
    example_batch,
    compile_fit_evaluate,
    K_fold_training,
    create_dataset_lite,
)

#### Create the dataset

In [ ]:
# refit the model with the best parameters on the whole dataset
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    batch_size=batch_size,
    shuffle=True,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    show_example_batch=True,
    verbose=0,
)

#### Train the model

In [ ]:
model = build_model(n_labels=n_labels, **best_params)

epochs = 100
patience = 20
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    optimizer=(
        tf.keras.optimizers.legacy.Adam(learning_rate=1e-3)
        if sys.platform == "darwin"
        else tf.keras.optimizers.Adam(learning_rate=1e-3)
    ),
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

## 2.3 Convolutional Neural Networks

Within this section, we will use Convolutional Neural Networks (CNNs) for the classification of audio files. To accomplish this, we will leverage the ESC-10 dataset, subjecting the audio to three distinct preprocessing methodologies. For each preprocessing technique, we will execute a comprehensive grid search, fine-tuning the parameters to identify optimal configurations. Importantly, the parameters under consideration will remain consistent across the various preprocessing strategies. Ultimately, our aim is to conduct a comparative analysis of the outcomes obtained from these different preprocessing routes. Subsequently, we will narrow our focus to the most promising model, employing the finest preprocessing methodology, and apply it to the ESC-50 dataset for further training and evaluation. This progression ensures that our efforts are guided by a systematic exploration of preprocessing variations, leading us to the optimal model for the subsequent stages of our investigation.

### 2.3.1 CNN - STFT Preprocessed Audio

In [ ]:
importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    create_dataset,
    compile_fit_evaluate,
    K_fold_training,
    create_dataset_lite,
)

#### Create the dataset ESC-10

In [ ]:
batch_size = 30

dataset, label = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing="STFT",
    ndim=3,
)

INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns))

#### Build the model

We are going to keep this function for sections 2.3.2, 2.3.3 and 2.3.4 to avoid repeating the code

In [ ]:
def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    kernel_size=(3, 3),
    activation="relu",
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    model = tf.keras.models.Sequential(
        [
            # Convolutional layer 1
            tf.keras.layers.Conv2D(
                n_units,
                kernel_size,
                strides=2,
                activation=activation,
                input_shape=INPUT_DIM,
                padding="same",
            ),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            # Convolutional layer 2
            tf.keras.layers.Conv2D(
                2 * n_units, (3, 3), strides=2, activation=activation, padding="same"
            ),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            # Convolutional layer 3
            tf.keras.layers.Conv2D(
                4 * n_units, (3, 3), activation=activation, padding="same"
            ),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            # Flatten the output of the previous layer
            tf.keras.layers.Flatten(),
            # Dense layer for classification
            tf.keras.layers.Dense(
                n_labels, activation="softmax"
            ),  # Assuming 10 classes for classification
        ],
        name="CNN",
    )

    if compile:
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin" or in_colab
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f"n_units {n_units}, activation {activation}, learning_rate {learning_rate}, kernel size {kernel_size}"
    )

    return model

#### Run a grid search to find the best params

In [ ]:
epochs = 40
patience = 5
params = {
    "INPUT_DIM": [INPUT_DIM],
    "n_units": [16, 32],
    "activation": ["relu", "tanh"],
    "learning_rate": [1e-3, 1e-4],
    "kernel_size": [(3, 3), (5, 5)],
}
K_fold = 4

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_STFT_231.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the best model on ESC-50

In [ ]:
# refit only the best model on ESC-50
seed = 42
tf.random.set_seed(seed)
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
preprocessing = "STFT"

train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    verbose=0,
    batch_size=batch_size,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT", "MEL", "MFCC" or None
    show_example_batch=True,
    ndim=3,
)

In [ ]:
# upload the best parameters
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_STFT_231.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

In [ ]:
# build the model
model = build_model(n_labels=n_labels, compile=False, **best_params)

epochs = 50
patience = 10  # early stopping patience
lr = best_params["learning_rate"]
model, hisotry, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=(
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    ),
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

In [ ]:
# show the weights of the first layer
visualize_the_weights(model, layer_number=0, n_filters=16, verbose=0)

In [ ]:
save_model_path = os.path.join(main_dir, "Saved_Models", "ESC50_simple_CNN_STFT")
# save the model
model.save(save_model_path, save_format="keras")

#### Train the best model on augmented data

In the data_augmentation notebook starting with ESC-50 we created 20 other equally large datasets, half of them with time masks and the other half with frequency masks. Now we try to train the best model resulting from gridsearch on the augmented dataset. 

Load the already trained simple CNN STFT model which was trained on ESC-50. 

In [ ]:
# Define the path to the saved model
saved_model_path = os.path.join(main_dir, "Saved_Models", "ESC50_simple_CNN_STFT")

# Load the model
loaded_model = tf.keras.models.load_model(saved_model_path)

Duplicate it so we will be able to compare the results

In [ ]:
save_model_path = os.path.join(
    main_dir, "Saved_Models", "ESC50_simple_CNN_STFT_Augmented"
)
# save the model
loaded_model.save(save_model_path, save_format="keras")

Upload the best parameters for the model to use into the build function

In [ ]:
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_STFT_231.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

We are going to use val and test set from the ESC-50 original dataset but train is going to be a masked_dataset

In [ ]:
# refit only the best model on ESC-50
seed = 42
tf.random.set_seed(seed)
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
preprocessing = "STFT"

train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    verbose=0,
    batch_size=batch_size,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT", "MEL", "MFCC" or None
    show_example_batch=True,
    ndim=3,
)

In [ ]:
# Around 12 min to run 50 epochs on one masked dataset (we have 20)
from Models.ann_utils import create_masked_dataset

# create a log of the training or load the existing one:
log_dir = os.path.join(main_dir, "Logs_Masked")

list_masked_dataset = os.listdir(
    os.path.join(main_dir, "Saved_Datasets", "masked_dataset")
)
if ".DS_Store" in list_masked_dataset:
    list_masked_dataset.remove(".DS_Store")

for dataset_name in list_masked_dataset:
    print(f"Working on {dataset_name}")

    # stuff with the log
    if not os.path.exists(log_dir):
        os.mkdir(log_dir)
    # if log dir is empty
    if not os.listdir(log_dir):
        # create dictionary with the name of the dataset as key and False as value
        dict_dataset = {dataset: False for dataset in list_masked_dataset}
        # save the best_params in pickle
        with open(
            os.path.join(main_dir, "Logs_Masked", "dict_masked_dataset.pickle"), "wb"
        ) as handle:
            pickle.dump(dict_dataset, handle, protocol=pickle.HIGHEST_PROTOCOL)
    else:
        with open(
            os.path.join(main_dir, "Logs_Masked", "dict_masked_dataset.pickle"), "rb"
        ) as handle:
            dict_dataset = pickle.load(handle)

    # start the real training
    if not dict_dataset[dataset_name]:
        print("Training the model on the masked dataset: ", dataset_name)
        # loading the masked dataset
        dataset_path = os.path.join(
            main_dir, "Saved_Datasets", "masked_dataset", dataset_name
        )
        masked_dataset = tf.data.Dataset.load(dataset_path)

        # preprocessing into train_masked
        train_masked = create_masked_dataset(
            dataset_path, batch_size=30, normalize=True, verbose=0
        )

        # load ESC50_simple_CNN_STFT_Augmented
        saved_model_path = os.path.join(
            main_dir, "Saved_Models", "ESC50_simple_CNN_STFT_Augmented"
        )
        loaded_model = tf.keras.models.load_model(saved_model_path)

        # compile, fit and evaluate
        epochs = 50
        patience = 10  # early stopping patience
        lr = best_params["learning_rate"]

        # last folder check
        s = sum(dict_dataset.values())
        if s == len(list_masked_dataset) - 1:
            show_history = True
            show_test_evaluation = True
            show_confusion_matrix = True
            listen_to_wrong = True
        else:
            show_history = False
            show_test_evaluation = False
            show_confusion_matrix = False
            listen_to_wrong = False

        # note that the dataframe df_ESC50 does not create any problem in this framework (it is not used)
        output = compile_fit_evaluate(
            df_ESC50,
            loaded_model,
            train_masked,
            val,
            test,
            label_names,
            epochs=epochs,
            patience=patience,
            loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=lr)
                if sys.platform == "darwin" or in_colab
                else tf.keras.optimizers.Adam(learning_rate=lr)
            ),
            metrics=["accuracy"],
            verbose=0,
            show_history=show_history,
            show_test_evaluation=show_test_evaluation,
            show_confusion_matrix=show_confusion_matrix,
            listen_to_wrong=listen_to_wrong,
        )

        if s == len(list_masked_dataset) - 1:
            model, hisotry, confusion_mtx, evaluation = (
                output[0],
                output[1],
                output[2],
                output[3],
            )
        else:
            model, hisotry = output[0], output[1]

        # save the model
        save_model_path = os.path.join(
            main_dir, "Saved_Models", "ESC50_simple_CNN_STFT_Augmented"
        )
        model.save(save_model_path, save_format="keras")

        # update the log
        dict_dataset[dataset_name] = True

        # save the log
        with open(
            os.path.join(main_dir, "Logs_Masked", "dict_masked_dataset.pickle"), "wb"
        ) as handle:
            pickle.dump(dict_dataset, handle, protocol=pickle.HIGHEST_PROTOCOL)

    else:
        print(f"The model has already been trained on {dataset_name}")

    print("Number of masked dataset trained: ", sum(dict_dataset.values()))

In [ ]:
# show the weights of the first layer
visualize_the_weights(model, layer_number=0, n_filters=16, verbose=0)

In [ ]:
# load the dictionary
with open(
    os.path.join(main_dir, "Logs_Masked", "dict_masked_dataset.pickle"), "rb"
) as handle:
    dict_dataset_see = pickle.load(handle)

dict_dataset_see  # see if all the datasets have been trained

### 2.3.2 CNN - MEL Preprocessed Audio

In [ ]:
importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    create_dataset,
    compile_fit_evaluate,
    K_fold_training,
    create_dataset_lite,
)

#### Create the dataset ESC-10

In [ ]:
batch_size = 30

dataset, label = create_dataset_lite(
    df_ESC10, batch_size=batch_size, preprocessing="MEL", ndim=3, verbose=0
)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=0)

#### Run a grid search to find the best params

In [ ]:
epochs = 40
patience = 5
params = {
    "INPUT_DIM": [INPUT_DIM],
    "n_units": [16, 32],
    "activation": ["relu", "tanh"],
    "learning_rate": [1e-3, 1e-4],
    "kernel_size": [(3, 3), (7, 7)],
}
K_fold = 4

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_MEL_232.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the best model on ESC-50

In [ ]:
# refit only the best model on ESC-50
seed = 42
tf.random.set_seed(seed)
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
preprocessing = "MEL"

train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    verbose=0,
    batch_size=batch_size,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT", "MEL", "MFCC" or None
    show_example_batch=True,
    ndim=3,
)

In [ ]:
# load the best parameters
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_MEL_232.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

In [ ]:
# build the model
model = build_model(n_labels=n_labels, compile=False, **best_params)

epochs = 50
patience = 10  # early stopping patience
lr = best_params["learning_rate"]
model, hisotry, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=(
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin"
        else tf.keras.optimizers.Adam(learning_rate=lr)
    ),
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

In [ ]:
# show the weights of the first layer
visualize_the_weights(model, layer_number=0, n_filters=16, verbose=0)

In [ ]:
save_model_path = os.path.join(main_dir, "Saved_Models", "ESC50_simple_CNN_MEL")
# save the model
model.save(save_model_path, save_format="keras")

### 2.3.3 CNN - MFCC Preprocessed Audio

In [ ]:
importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    create_dataset,
    compile_fit_evaluate,
    K_fold_training,
    create_dataset_lite,
)

#### Create the dataset ESC-10

We tested also without delta or delta-delta coefficients and the best results were achieved including both of them in the preprocessing.

In [ ]:
batch_size = 30
preprocessing = "MFCC"
delta = True
delta_delta = True
dataset, label = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing=preprocessing,
    delta=delta,
    delta_delta=delta_delta,
    ndim=3,
)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=0)

#### Run a grid search to find the best params

In [ ]:
epochs = 40
patience = 5
params = {
    "INPUT_DIM": [INPUT_DIM],
    "n_units": [64, 128],
    "activation": ["tanh"],
    "learning_rate": [1e-3, 1e-4],
    "kernel_size": [(3, 3), (7, 7)],
}
# params = {'INPUT_DIM' : [INPUT_DIM],'n_units':[64], 'activation':['relu'], 'learning_rate':[1e-3, 1e-4], 'kernel_size':[(3,3)]}
K_fold = 4

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_MFCC_233.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

Here are the results of the gridsearches without delta or delta-delta

In [ ]:
# result.to_csv(os.path.join(main_dir,'Saved_Models','Simple_CNN_MFCC_all_delta_grid_search_results.csv'), index =True)
display(
    pd.read_csv(
        os.path.join(
            main_dir, "Saved_Models", "Simple_CNN_MFCC_no_delta_grid_search_results.csv"
        ),
        index_col=0,
    )
)
display(
    pd.read_csv(
        os.path.join(
            main_dir,
            "Saved_Models",
            "Simple_CNN_MFCC_only_first_delta_grid_search_results.csv",
        ),
        index_col=0,
    )
)
display(
    pd.read_csv(
        os.path.join(
            main_dir,
            "Saved_Models",
            "Simple_CNN_MFCC_all_delta_grid_search_results.csv",
        ),
        index_col=0,
    )
)

#### Train the best model on ESC-50

In [ ]:
# refit only the best model
seed = 42
tf.random.set_seed(seed)
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
preprocessing = "MFCC"
delta = True
delta_delta = True

train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    verbose=0,
    batch_size=batch_size,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT", "MEL", "MFCC" or None
    delta=delta,
    delta_delta=delta_delta,
    show_example_batch=True,
    ndim=3,
)

In [ ]:
# load the best parameters
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_MFCC_233.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

In [ ]:
# build the model
model = build_model(n_labels=n_labels, compile=False, **best_params)

epochs = 50
patience = 10  # early stopping patience
lr = best_params["learning_rate"]
model, hisotry_supervised, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=(
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin"
        else tf.keras.optimizers.Adam(learning_rate=lr)
    ),
    metrics=["accuracy"],  # ,'CategoricalAccuracy'],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

In [ ]:
# show the weights of the first layer
visualize_the_weights(model, layer_number=0, n_filters=16, verbose=0)

In [ ]:
save_model_path = os.path.join(
    main_dir, "Saved_Models", "ESC50_simple_CNN_MFCC_all_delta"
)
# save the model
model.save(save_model_path, save_format="keras")